In [1]:
import json
import os
from typing import Dict, List
import pandas as pd
import numpy as np
import time
import tushare as ts
import openai
import re
import hashlib

In [ ]:

class InformationBase:
    def __init__(self, filepath: str = "information_base.json", max_memory_size: int = 15):
        self.filepath = filepath
        self.max_memory_size = max_memory_size
        self.memory = self._load_or_init()

    def _load_or_init(self) -> Dict[str, List[str]]:
        """加载或初始化信息库文件"""
        if os.path.exists(self.filepath):
            with open(self.filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        else:
            initial_data = {
                "recommended_directions": ["尝试量价结合计算波动率。"],
                "prohibited_directions": ["禁止将绝对价格直接作为因子。"]
            }
            self._save(initial_data)
            return initial_data

    def _save(self, data: Dict = None):
        """持久化保存到 JSON"""
        data_to_save = data if data else self.memory
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(data_to_save, f, ensure_ascii=False, indent=4)

    def update_experience(self, action: str, content: str):
        """
        更新记忆，并自动维护队列长度 (FIFO)
        action: 'add_recommended', 'add_prohibited', 'add_failed_attempt'
        """
        key_map = {
            "add_recommended": "recommended_directions",
            "add_prohibited": "prohibited_directions"
        }
        target_key = key_map.get(action)
        if target_key:
            self.memory[target_key].append(content)
            # 截断早期记忆，防止 LLM Context 爆炸
            self.memory[target_key] = self.memory[target_key][-self.max_memory_size:]
            self._save()

    def get_prompt_summary(self) -> str:
        """生成供 LLM 读取的字符串摘要"""
        return json.dumps(self.memory, ensure_ascii=False, indent=2)

In [3]:
class FactorBase:
    def __init__(self, metadata_path: str = "factor_base.json", data_dir: str = "./factor_data/"):
        self.metadata_path = metadata_path
        self.data_dir = data_dir
        os.makedirs(self.data_dir, exist_ok=True)
        self.factors_metadata = self._load_or_init()

    def _load_or_init(self) -> List[Dict]:
        if os.path.exists(self.metadata_path):
            with open(self.metadata_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        return[]

    def _save_metadata(self):
        with open(self.metadata_path, 'w', encoding='utf-8') as f:
            json.dump(self.factors_metadata, f, ensure_ascii=False, indent=4)

    def add_factor(self, formula: str, metrics: Dict, factor_df: pd.DataFrame):
        """入库新因子：保存元数据并落盘数据"""
        factor_id = f"alpha_{len(self.factors_metadata) + 1:04d}"
        
        # 1. 保存元数据
        new_record = {
            "factor_id": factor_id,
            "formula": formula,
            "metrics": metrics
        }
        self.factors_metadata.append(new_record)
        self._save_metadata()
        
        # 2. 保存计算好的 DataFrame 到本地 (使用 parquet 极速读写)
        file_path = os.path.join(self.data_dir, f"{factor_id}.parquet")
        factor_df.to_parquet(file_path)

    def get_all_factor_data(self) -> Dict[str, pd.DataFrame]:
        """加载所有已有因子的数据，用于正交化/相关性检验"""
        loaded_data = {}
        for factor in self.factors_metadata:
            fid = factor["factor_id"]
            file_path = os.path.join(self.data_dir, f"{fid}.parquet")
            if os.path.exists(file_path):
                loaded_data[fid] = pd.read_parquet(file_path)
        return loaded_data

    def get_prompt_summary(self) -> str:
        """只暴露公式给 LLM 避免过长"""
        formulas = [f["formula"] for f in self.factors_metadata]
        return "\n".join(formulas) if formulas else "暂无因子"

In [ ]:


class EvaluationEngine:
    def __init__(self, stock_data: dict, forward_returns: pd.DataFrame, factor_base, benchmark_returns: pd.Series):
        self.stock_data = stock_data
        self.forward_returns = forward_returns
        self.factor_base = factor_base
        self.benchmark_returns = benchmark_returns # [新增] 注入沪深300基准收益率
        
        self.safe_env = self._register_operators()
        self.safe_env.update(self.stock_data)
        
    def _register_operators(self) -> dict:
        """
        全量化因子算子库 
        """
        # --- 辅助函数：安全算术 ---
        def safe_log(x): return np.log(x.abs().replace(0, np.nan))
        def safe_inv(x): return 1 / x.replace(0, np.nan)
        def safe_sqrt(x): return np.sqrt(x.abs())
        
        # --- 辅助函数：时序回归 (向量化计算提升速度) ---
        def roll_slope(x, d):
            # 使用 numpy 的 polyfit 提速，若含 NaN 则返回 NaN
            return x.rolling(window=d).apply(lambda v: np.polyfit(np.arange(d), v, 1)[0] if not np.isnan(v).any() else np.nan, raw=True)
            
        def roll_rsquare(x, d):
            return x.rolling(window=d).apply(lambda v: np.corrcoef(np.arange(d), v)[0, 1]**2 if not np.isnan(v).any() else np.nan, raw=True)
            
        def roll_resi(x, d):
            def _resi(v):
                if np.isnan(v).any(): return np.nan
                p = np.polyfit(np.arange(d), v, 1)
                return v[-1] - (p[0] * (d - 1) + p[1])
            return x.rolling(window=d).apply(_resi, raw=True)

        return {
            # 1. Arithmetic (算术运算 - 元素级)
            'Add': lambda x, y: x + y,
            'Sub': lambda x, y: x - y,
            'Mul': lambda x, y: x * y,
            'Div': lambda x, y: x / y.replace(0, np.nan),
            'Neg': lambda x: -x,
            'Abs': lambda x: x.abs(),
            'Sign': lambda x: np.sign(x),                 
            'Max2': lambda x, y: np.maximum(x, y),      
            'Min2': lambda x, y: np.minimum(x, y), 
            'Log': safe_log,
            'SignedPower': lambda x, e: np.sign(x) * (x.abs() ** e),
            'Power': lambda x, e: x ** e,
            'Inv': safe_inv,
            'Sqrt': safe_sqrt,
            'Square': lambda x: x ** 2,
            'Exp': lambda x: np.exp(x),
            'Tanh': lambda x: np.tanh(x),

            # 2. Statistical (统计特性 - 滚动窗口)
            'Mean': lambda x, d: x.rolling(window=d).mean(),
            'Std': lambda x, d: x.rolling(window=d).std(),
            'Var': lambda x, d: x.rolling(window=d).var(),
            'Skew': lambda x, d: x.rolling(window=d).skew(),
            'Kurt': lambda x, d: x.rolling(window=d).kurt(),
            'Med': lambda x, d: x.rolling(window=d).median(),
            'Sum': lambda x, d: x.rolling(window=d).sum(),
            'Product': lambda x, d: x.rolling(window=d).apply(np.prod, raw=True),

            # 3. Time-series (时序模式 - 滚动窗口)
            'Delay': lambda x, d: x.shift(d),
            'Delta': lambda x, d: x.diff(d),
            'TsRank': lambda x, d: x.rolling(window=d).rank(pct=True),
            'TsMax': lambda x, d: x.rolling(window=d).max(),
            'TsMin': lambda x, d: x.rolling(window=d).min(),
            'TsArgMax': lambda x, d: x.rolling(window=d).apply(np.argmax, raw=True),
            'TsArgMin': lambda x, d: x.rolling(window=d).apply(np.argmin, raw=True),
            'TsDecay': lambda x, d: x.rolling(window=d).apply(
                lambda v: np.dot(v, np.arange(1, d + 1)) / np.arange(1, d + 1).sum(), raw=True
            ), # 线性加权衰减
            'Cov': lambda x, y, d: x.rolling(window=d).cov(y),   
            'Corr': lambda x, y, d: x.rolling(window=d).corr(y), 
            
            # 4. Cross-sectional (截面变换)
            'CsRank': lambda x: x.rank(axis=1, pct=True),
            'Scale': lambda x, a=1: x.div(x.abs().sum(axis=1), axis=0) * a, # 标准化使绝对值和为a

            # 5. Smoothing (平滑/趋势提取)
            'SMA': lambda x, d: x.rolling(window=d).mean(),
            'EMA': lambda x, d: x.ewm(span=d, adjust=False).mean(),
            'WMA': lambda x, d: x.rolling(window=d).apply(
                lambda v: np.dot(v, np.arange(1, d + 1)) / np.arange(1, d + 1).sum(), raw=True
            ),

            # 6. Regression (回归提取 - 滚动时间窗口上的线性回归)
            'Slope': roll_slope,
            'Rsquare': roll_rsquare,
            'Resi': roll_resi,

            # 7. Logical (逻辑判断 - 状态切换)
            'IfElse': lambda cond, x, y: pd.DataFrame(np.where(cond, x, y), index=x.index, columns=x.columns),
            'Greater': lambda x, y: x > y,
            'Less': lambda x, y: x < y,
            'GreaterEqual': lambda x, y: x >= y,
            'LessEqual': lambda x, y: x <= y,
            'And': lambda x, y: x & y,
            'Or': lambda x, y: x | y,
            'Eq': lambda x, y: x == y,
            'Ne': lambda x, y: x != y
        }

    def _calculate_factor(self, formula: str) -> pd.DataFrame:
        """执行表达式树（生产环境应替换为 AST Parser）"""
        return eval(formula, {"__builtins__": {}}, self.safe_env)

    def evaluate(self, formula: str) -> tuple[bool, dict, pd.DataFrame]:
        """
        核心流水线：传入公式，执行完整的工业级量化因子检验标准
        返回 (是否通过, 评测指标字典, 计算出的因子DataFrame)
        """
        try:
            # 1. 计算因子值
            factor_df = self._calculate_factor(formula)
            
            # ==========================================
            # A. 预测能力指标 (IC 相关)
            # ==========================================
            # 计算每日 Rank IC (因子值与未来收益率的截面秩相关系数)
            ic_series = factor_df.corrwith(self.forward_returns, axis=1, method='spearman').dropna()
            
            if len(ic_series) < 2:
                raise ValueError("有效 IC 样本量不足")

            ic_mean = float(ic_series.mean())
            ic_std = float(ic_series.std())
            
            # IR (信息比率) 和 t-value (t检验值，判断显著性)
            ir = ic_mean / ic_std if ic_std != 0 else 0.0
            t_value = ic_mean / (ic_std / np.sqrt(len(ic_series))) if ic_std != 0 else 0.0
            
            # IC 胜率 (动态方向判断：看主导方向胜率)
            if ic_mean > 0:
                ic_win_rate = float((ic_series > 0).mean())
            else:
                ic_win_rate = float((ic_series < 0).mean())
                
            # 因子衰减 Decay (分别计算对 T+1, T+2, T+3 的预测能力)
            # shift(-1) 代表拿今天的因子去预测后天的收益率
            ic_decay_2 = factor_df.corrwith(self.forward_returns.shift(-1), axis=1, method='spearman').mean()
            ic_decay_3 = factor_df.corrwith(self.forward_returns.shift(-2), axis=1, method='spearman').mean()
            decay_tuple = (round(ic_mean, 3), round(float(ic_decay_2), 3), round(float(ic_decay_3), 3))

            # ==========================================
            # B. 收益率与分组指标 (分组回测)
            # ==========================================
            # 将每日截面转化为 0-1 的百分位，用于分 5 组 (Quintiles)
            ranks = factor_df.rank(axis=1, pct=True)
            q_rets =[]
            
            for i in range(5):
                # 提取各分组的掩码: Q1(0~20%) ... Q5(80~100%)
                mask = (ranks > i * 0.2) & (ranks <= (i + 1) * 0.2)
                # 计算该组内所有股票每天的平均收益
                group_daily_ret = self.forward_returns[mask].mean(axis=1)
                q_rets.append(group_daily_ret)
                
            # 计算组别单调性 Monotonicity (1-5组与组平均收益率的线性相关性)
            q_mean_rets =[q.mean() for q in q_rets]
            monotonicity = float(np.corrcoef(q_mean_rets,[1, 2, 3, 4, 5])[0, 1])
            if np.isnan(monotonicity): monotonicity = 0.0
                
            # 计算多空每日收益序列 (做多最好的一组，做空最差的一组)
            if ic_mean > 0:
                ls_daily_ret = q_rets[4] - q_rets[0]  # 正向因子：多 Q5 空 Q1
            else:
                ls_daily_ret = q_rets[0] - q_rets[4]  # 反向因子：多 Q1 空 Q5
                
            ls_daily_ret = ls_daily_ret.fillna(0)
            
            # 2. 对齐基准日收益率 (处理潜在的日期缺失)
            bench_ret = self.benchmark_returns.reindex(ls_daily_ret.index).fillna(0)

            # 多空年化收益率、年化波动率、多空夏普比率 (假设一年252交易日)
            excess_daily_ret = ls_daily_ret - bench_ret
            excess_ann_ret = float(excess_daily_ret.mean() * 252)
            ls_ann_ret = float(ls_daily_ret.mean() * 252)
            ls_ann_vol = float(ls_daily_ret.std() * np.sqrt(252))
            ls_ann_sharpe = excess_ann_ret / ls_ann_vol if ls_ann_vol != 0 else 0.0
            
            # 最大回撤 Max Drawdown (基于多空对冲净值计算)
            cum_net_value = (1 + ls_daily_ret).cumprod()
            running_max = cum_net_value.cummax()
            drawdowns = (cum_net_value - running_max) / running_max
            max_drawdown = float(drawdowns.min())

            # ==========================================
            # C. 落地约束与正交化约束
            # ==========================================
            # 换手率 (Rank 值的日均绝对变化幅度，代替精确换手率)
            turnover = float(factor_df.rank(axis=1, pct=True).diff().abs().mean(axis=1).mean())
            
            # 计算与已有因子库的最大相关性 (避免重复造轮子)
            existing_factors = self.factor_base.get_all_factor_data()
            max_corr = 0.0
            for ext_df in existing_factors.values():
                corr = factor_df.corrwith(ext_df, axis=1).mean()
                if abs(corr) > max_corr:
                    max_corr = abs(corr)
                    
            # ==========================================
            # D. 输出封装与准入判定 (非常严格)
            # ==========================================
            metrics = {
                "IC": round(ic_mean, 4),
                "IR": round(ir, 4),
                "IC_WinRate": round(ic_win_rate, 4),
                "t_value": round(t_value, 2),
                "EXCESS_Return":round(excess_ann_ret,4),
                "LS_Return": round(ls_ann_ret, 4),
                "LS_Sharpe": round(ls_ann_sharpe, 4),
                "Max_Drawdown": round(max_drawdown, 4),
                "Monotonicity": round(monotonicity, 4),
                "Turnover": round(turnover, 4),
                "Max_Correlation": round(max_corr, 4),
                "Decay(T1-T3)": decay_tuple
            }
            
            # 严苛的综合判断条件 (六边形战士才能入库)
            is_success = (
                (abs(ic_mean) > 0.03) and          # 预测能力及格线
                (abs(t_value) > 2.0) and           # 必须具备统计学显著性(置信度95%以上)
                (turnover < 0.3) and               # 拒绝高频噪音，换手不能吃光利润
                (max_corr < 0.6)                   # 必须是对抗红海的正交新 Alpha
            )
            
            return is_success, metrics, factor_df

        except Exception as e:
            # 捕获公式异常（如未定义的变量、极端值溢出等）
            return False, {"Error": str(e)}, None

In [ ]:
class FactorMiningAgent:
    def __init__(self, info_base: InformationBase, factor_base: FactorBase, eval_engine: EvaluationEngine, api_key: str, logic=False):
        self.info_base = info_base
        self.factor_base = factor_base
        self.eval_engine = eval_engine
        self.client = openai.Client(api_key=api_key,base_url="https://api.chatanywhere.tech/v1")
        self.logic=logic

    def generate(self) -> str:
        """Step 1 & 2: 读取外部记忆，生成新公式"""
        if self.logic:
            prompt = f"""
            你是一个顶尖的量化因子挖掘专家。请根据信息库生成一个全新的 Alpha 因子公式。
            
            【可用数据列表】
            Close, Open, High, Low, Volume, Amount

            【可用算子库】
            1. 算术: Add(x,y), Sub(x,y), Mul(x,y), Div(x,y), Neg(x), Abs(x), Sign(x), Max2(x,y), Min2(x,y), Log(x), SignedPower(x,e), Power(x,e), Sqrt(x)
            2. 统计(滚动d天): Mean(x,d), Std(x,d), Skew(x,d), Kurt(x,d), Med(x,d), Sum(x,d)
            3. 时序与相关(滚动d天): Delay(x,d), Delta(x,d), TsRank(x,d), TsMax(x,d), TsMin(x,d), TsArgMax(x,d), TsDecay(x,d), Cov(x,y,d), Corr(x,y,d)
            4. 截面: CsRank(x), Scale(x)
            5. 平滑: SMA(x,d), EMA(x,d), WMA(x,d)
            6. 时序回归(滚动d天): Slope(x,d), Rsquare(x,d), Resi(x,d)
            7. 逻辑(返回条件判断): IfElse(cond, x, y), Greater(x,y), Less(x,y), And(x,y), Or(x,y), Eq(x,y)

            【公式示例】
            - 影线逻辑: Div(Sub(Min2(Open, Close), Low), Add(Sub(High, Low), 0.0001))
            - 量价协方差反转: Neg(Mul(TsRank(Cov(TsRank(Close,18), TsRank(Volume,18), 10), 18), CsRank(Returns)))
            - 状态切换: IfElse(Greater(Kurt(Returns, 24), 3), Neg(Resi(Close, 6)), Neg(Slope(Close, 12)))
            【已有因子】（禁止重复）:\n{self.factor_base.get_prompt_summary()}
            【信息库经验】（必须遵循）:\n{self.info_base.get_prompt_summary()}
            
            仅输出一行纯文本的数学公式。
            """
            response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=1
        )
            return response.choices[0].message.content.strip()
        
        else:
            prompt = f"""
        你是一位顶尖的量化因子挖掘专家。你的任务是结合市场微观结构、行为金融学或传统的量价逻辑，构建一个全新的、具有强逻辑支撑的 Alpha 因子。

        【可用数据列表】
        Close, Open, High, Low, Volume, Amount

        【可用算子库】
        1. 算术: Add(x,y), Sub(x,y), Mul(x,y), Div(x,y), Neg(x), Abs(x), Sign(x), Max2(x,y), Min2(x,y), Log(x), SignedPower(x,e), Power(x,e), Sqrt(x)
        2. 统计(滚动d天): Mean(x,d), Std(x,d), Skew(x,d), Kurt(x,d), Med(x,d), Sum(x,d)
        3. 时序与相关(滚动d天): Delay(x,d), Delta(x,d), TsRank(x,d), TsMax(x,d), TsMin(x,d), TsArgMax(x,d), TsDecay(x,d), Cov(x,y,d), Corr(x,y,d)
        4. 截面处理(消除市场beta): CsRank(x), Scale(x)
        5. 平滑(降低换手): SMA(x,d), EMA(x,d), WMA(x,d)
        6. 时序回归(滚动d天): Slope(x,d), Rsquare(x,d), Resi(x,d)
        7. 逻辑(状态切换): IfElse(cond, x, y), Greater(x,y), Less(x,y), And(x,y), Or(x,y), Eq(x,y)

            【公式示例】
            - 影线逻辑: Div(Sub(Min2(Open, Close), Low), Add(Sub(High, Low), 0.0001))
            - 量价协方差反转: Neg(Mul(TsRank(Cov(TsRank(Close,18), TsRank(Volume,18), 10), 18), CsRank(Returns)))
            - 状态切换: IfElse(Greater(Kurt(Returns, 24), 3), Neg(Resi(Close, 6)), Neg(Slope(Close, 12)))
            【已有因子】（禁止重复）:\n{self.factor_base.get_prompt_summary()}
            【信息库经验】（必须遵循）:\n{self.info_base.get_prompt_summary()}
            

        【输出格式要求】
        请使用严谨的量化思维进行逐步推理，并严格返回如下 JSON 格式（不要附带 markdown 等其他字符）：
        {{
            "financial_hypothesis": "阐述该因子的金融逻辑与微观原理。",
            "operator_mapping": "说明为什么选择某些具体的算子组合来实现上述假说，特别是如何处理量纲、极端值和降低换手率。",
            "formula": "最终的一行纯文本数学公式（仅包含公式本身，禁止任何换行或额外字符）。"
        }}
        """
        raw_content = ""
        try:
            response = self.client.chat.completions.create(
                model="deepseek-chat",
                messages=[{"role": "user", "content": prompt}],
                temperature=1  # 挖掘阶段保持一定的创造力
            )
            raw_content = response.choices[0].message.content.strip()
            
            # 【核心：强力 JSON 清洗防御】
            cleaned_content = raw_content.replace('```json', '').replace('```', '').strip()
            
            # 用正则暴力提取大括号内的内容
            match = re.search(r'\{.*\}', cleaned_content, re.DOTALL)
            if match:
                cleaned_content = match.group(0)
                
            # 去除可能导致解析崩溃的非法换行/制表符，但保留公式的连贯性
            cleaned_content = cleaned_content.replace('\n', ' ').replace('\r', '').replace('\t', ' ')
            
            # 解析 JSON
            parsed_data = json.loads(cleaned_content)
            
            # 提取字段
            formula = parsed_data.get("formula", "").strip()
            hypothesis = parsed_data.get("financial_hypothesis", "未提供逻辑假说")
            mapping = parsed_data.get("operator_mapping", "未提供算子映射")
            
            if not formula:
                raise ValueError("LLM 未在 JSON 中返回 formula 字段")
                
            return formula
            
        except json.JSONDecodeError:
            print(f"\n⚠️ [生成跳过] LLM 输出 JSON 格式异常。原始输出片段: {raw_content[:150]}...")
            return None, None, None
        except Exception as e:
            print(f"\n⚠️ [生成跳过] 因子生成阶段发生异常: {e}")
            return None, None, None

    def distill(self, formula: str, metrics: dict, is_success: bool):
        """Step 4: 基于评测结果进行反思，并更新信息库"""
        status = "成功" if is_success else "失败"
        if self.logic:
            prompt = f"""
            因子挖掘{status}。公式: {formula}。结果: {json.dumps(metrics)}
            请作为量化专家分析原因，并输出严格的 JSON 指令以更新经验库：
            {{
                "action": "add_recommended" 或 "add_prohibited" 
                "content": "具体的一句反思总结"
            }}
            注意：请绝对只输出这一个 JSON 字典！不要使用 markdown 语法(不要 ```json)，不要换行控制符，不要解释！反思的action严格参照是否成功！
            """
            
        else:
            prompt = f"""
        你是一位资深的量化研究员。你正在对一个新挖掘的 Alpha 因子进行回测结果的回顾与反思。
        
        【测试目标】
        公式: {formula}
        回测结果状态: {status}
        详细指标 : {json.dumps(metrics)}
        
        【任务要求】
        请基于以上量化指标进行深度的归因分析，并提取一条可泛化的经验/规则，更新到因子挖掘信息库中。
        1. 若成功：分析是哪种金融逻辑（如动量反转、波动率调整、量价背离等）或算子组合起到了关键作用。
        2. 若失败：诊断失败的根本原因（如：换手率过高导致摩擦成本大、过度拟合、线性相关性太高、算子量纲不匹配等）。
        
        【输出格式要求】
        严格输出一个合法的 JSON 对象，不要包含任何 markdown 语法（不要 ```json），不要换行控制符，不要解释！反思的action严格参照是否成功！格式如下：
        {{
            "action": "add_recommended" 或者 "add_prohibited",
            "content": "提炼出的一句高度精炼的量化经验法则（例如：'避免对高频量价相关性直接求和，需先进行截面去均值处理'）。"
        }}
        """
        try:
            response = self.client.chat.completions.create(
                model="deepseek-chat",  
                messages=[{"role": "user", "content": prompt}],
                temperature=1
            )
            raw_content = response.choices[0].message.content.strip()
            
            # 【清洗防御 1】：去除 Markdown 标记
            cleaned_content = raw_content.replace('```json', '').replace('```', '').strip()
            
            # 【清洗防御 2】：使用正则表达式强行提取大括号 {} 里的内容，防止 LLM 前后说废话
            match = re.search(r'\{.*\}', cleaned_content, re.DOTALL)
            if match:
                cleaned_content = match.group(0)
            
            # 【清洗防御 3】：去除可能导致 control character 报错的非法换行/制表符
            cleaned_content = cleaned_content.replace('\n', ' ').replace('\r', '').replace('\t', ' ')
            
            # 尝试解析
            instruction = json.loads(cleaned_content)
            
            action = instruction.get("action")
            content = instruction.get("content")
            
            if action and content:
                self.info_base.update_experience(action, content)
                # print(f"  [反思记录] {action}: {content}") # 如果需要看反思内容可以取消注释
                
        except json.JSONDecodeError as e:
            # 如果解析依然失败，打印错误，但使用 pass 跳过，保证循环继续
            print(f"\n⚠️ [解析跳过] LLM 输出格式异常无法解析。原始输出: {raw_content}")
        except Exception as e:
            # 捕获其他任何网络断开或 API 错误
            print(f"\n⚠️ [网络/API 异常] 反思阶段出错: {e}")

    def run_ralph_loop(self, max_iterations: int = 100):
        """核心无状态迭代循环"""
        for i in range(max_iterations):
            print(f"\n--- 启动第 {i+1} 轮迭代 ---")
            
            # 1. 生成因子
            formula = self.generate()
            print(f"Agent 生成公式: {formula}")
            
            # 2. 客观评测
            is_success, metrics, factor_df = self.eval_engine.evaluate(formula)
            print(f"评测结果: 是否成功={is_success}, 指标={metrics}")
            
            # 3. 提炼反思
            self.distill(formula, metrics, is_success)
            
            # 4. 入库结算
            if is_success and factor_df is not None:
                self.factor_base.add_factor(formula, metrics, factor_df)
                print(f"✅ 因子入库成功！ID 记录已更新。")
                
            time.sleep(1) # 避免 API 限流

In [ ]:
import os
import time
import hashlib
import pandas as pd
import tushare as ts

class TushareDataLoader:
    def __init__(self, token: str, start_date: str, end_date: str, stock_id=None, cache_dir: str = "./data_cache/"):
        self.token = token
        self.start_date = start_date
        self.end_date = end_date
        self.cache_dir = cache_dir
        
        # 兼容不同类型的 stock_id 传入格式
        if stock_id is None:
            self.stockid =[]
        elif isinstance(stock_id, str):
            self.stockid = [stock_id]
        else:
            self.stockid = stock_id

        ts.set_token(self.token)
        self.pro = ts.pro_api()
        os.makedirs(self.cache_dir, exist_ok=True)

    def fetch_daily_data(self) -> tuple:
        """
        按交易日循环获取全市场日线数据及复权因子，并生成【未复权】与【后复权】双路宽表矩阵。
        :return: (unadj_data: dict, post_adj_data: dict)
        """
        # 构建带特征哈希的缓存文件名，防止不同股票池缓存冲突
        if self.stockid:
            code_ident = hashlib.md5("_".join(sorted(self.stockid)).encode()).hexdigest()[:8]
            cache_file = os.path.join(self.cache_dir, f"custom_{code_ident}_{self.start_date}_{self.end_date}.parquet")
        else:
            cache_file = os.path.join(self.cache_dir, f"all_market_{self.start_date}_{self.end_date}.parquet")
        
        # ==========================================
        # 第一步：缓存读取 或 交易日历循环拉取
        # ==========================================
        if os.path.exists(cache_file):
            print(f"命中本地缓存: {cache_file}，正在使用 PyArrow 飞速加载全市场数据...")
            all_data = pd.read_parquet(cache_file)
        else:
            print(f"未命中缓存，正在获取 {self.start_date} 到 {self.end_date} 的交易日历...")
            # 获取区间内所有的交易日
            cal_df = self.pro.trade_cal(exchange='SSE', start_date=self.start_date, end_date=self.end_date, is_open=1)
            trade_dates = cal_df['cal_date'].tolist()
            
            if not trade_dates:
                raise ValueError(f"在 {self.start_date} 到 {self.end_date} 期间，未找到有效交易日！")

            print(f"共获取到 {len(trade_dates)} 个交易日，开始按日循环拉取量价与复权因子...")
            all_data_list =[]
            
            for i, date_str in enumerate(trade_dates):
                try:
                    # 1. 拉取当日所有股票的数据
                    df_daily = self.pro.daily(trade_date=date_str)
                    # 2. 拉取当日复权因子
                    df_adj = self.pro.adj_factor(trade_date=date_str)
                    
                    if not df_daily.empty and not df_adj.empty:
                        # 融合复权因子
                        df_adj = df_adj[['ts_code', 'adj_factor']]
                        df_merged = pd.merge(df_daily, df_adj, on='ts_code', how='left')
                        
                        # 如果指定了目标股票池，进行截断过滤
                        if self.stockid:
                            df_merged = df_merged[df_merged['ts_code'].isin(self.stockid)]
                            
                        all_data_list.append(df_merged)

                    # 严格控制休眠防封，按天拉取每天调用2个接口
                    time.sleep(0.12) 
                    
                    if (i + 1) % 100 == 0:
                        print(f"进度：已拉取 {i + 1} / {len(trade_dates)} 个交易日的数据...")
                        
                except Exception as e:
                    print(f"  [x] 下载 {date_str} 截面数据失败: {e}")
                    time.sleep(1) # 遇到网络波动稍微多等一会
                    
            print("\n正在拼接全市场数据表...")
            all_data = pd.concat(all_data_list, ignore_index=True)
            all_data['trade_date'] = pd.to_datetime(all_data['trade_date'])
            all_data.sort_values(['trade_date', 'ts_code'], inplace=True)
            
            all_data.to_parquet(cache_file, engine='pyarrow')
            print("全股票数据下载完成并已安全落盘 (Parquet格式)！\n")

        # ==========================================
        # 第二步：数据透视（长表转宽表矩阵）及 停牌逻辑处理
        # ==========================================
        print("正在构建量化计算引擎数据矩阵 (进行矩阵透视及停牌清洗)...")
        pivot_dict = {}
        cols_to_pivot =['open', 'high', 'low', 'close', 'vol', 'amount', 'adj_factor']
        
        for col in cols_to_pivot:
            if col not in all_data.columns: continue
            pivot_df = all_data.pivot(index='trade_date', columns='ts_code', values=col)
            pivot_df.sort_index(inplace=True)
            pivot_dict[col] = pivot_df

        # 重点逻辑：区分特征进行不同的缺失值(停牌)填充
        # 1. 价格和复权因子类 -> 停牌沿用昨收（向下填充 ffill）
        price_cols =['open', 'high', 'low', 'close', 'adj_factor']
        for col in price_cols:
            if col in pivot_dict:
                pivot_dict[col] = pivot_dict[col].ffill()
                
        # 2. 交易热度类 -> 停牌真实成交量/成交额为 0 (填充 0)
        vol_cols = ['vol', 'amount']
        for col in vol_cols:
            if col in pivot_dict:
                pivot_dict[col] = pivot_dict[col].fillna(0)

        # ==========================================
        # 第三步：计算双通道（复权与未复权）数据字典
        # ==========================================
        unadj_data = {}
        post_adj_data = {}
        
        # 组装未复权字典
        unadj_data['Open']       = pivot_dict.get('open')
        unadj_data['High']       = pivot_dict.get('high')
        unadj_data['Low']        = pivot_dict.get('low')
        unadj_data['Close']      = pivot_dict.get('close')
        unadj_data['Volume']     = pivot_dict.get('vol')
        unadj_data['Amount']     = pivot_dict.get('amount')
        unadj_data['Adj_Factor'] = pivot_dict.get('adj_factor') # 保留因子备用

        # 组装后复权字典：后复权价格 = 真实价格 * 复权因子
        adj_f = pivot_dict.get('adj_factor')
        if adj_f is not None:
            post_adj_data['Open']   = pivot_dict['open'] * adj_f
            post_adj_data['High']   = pivot_dict['high'] * adj_f
            post_adj_data['Low']    = pivot_dict['low'] * adj_f
            post_adj_data['Close']  = pivot_dict['close'] * adj_f
            post_adj_data['Volume'] = pivot_dict['vol'] 
            post_adj_data['Amount'] = pivot_dict['amount'] * adj_f     # 真实资金流水不需要复权
        
        return unadj_data, post_adj_data

    def build_forward_returns(self, close_df: pd.DataFrame, periods: int = 1) -> pd.DataFrame:
        """
        构建未来收益率矩阵 (用于 IC 测试)
        【注意】：参数 close_df 必须传入 `post_adj_data['Close']` (后复权收盘价)，
        否则分红除权日收益率会出现异常断崖。
        """
        forward_returns = close_df.shift(-periods) / close_df - 1.0
        return forward_returns
    
    def fetch_benchmark_returns(self, ts_code='000300.SH') -> pd.Series:
        """获取沪深300作为基准的 T+1 期收益率序列"""
        print(f"正在拉取基准指数 {ts_code} 数据...")
        
        df_index = self.pro.index_daily(ts_code=ts_code, start_date=self.start_date, end_date=self.end_date)
        if df_index is None or df_index.empty:
             raise ValueError("获取指数基准数据失败，请检查网络或权限！")
             
        df_index['trade_date'] = pd.to_datetime(df_index['trade_date'])
        df_index.sort_values('trade_date', inplace=True)
        df_index.set_index('trade_date', inplace=True)
        
        # 保证与股票端 future return 一样使用 shift(-1)
        bench_fwd_ret = df_index['close'].shift(-1) / df_index['close'] - 1.0
        
        return bench_fwd_ret

In [ ]:
if __name__ == "__main__":
    # 配置你的 Token 和 API Key
    TUSHARE_TOKEN = "1067ed1160b1dedeca68f122160aca415393dfd50f9f49ccbd1d0580"   # 去 Tushare 官网获取
    OPENAI_API_KEY = "sk-CqqsrgnNF94QRUDkpWmuC8NvVU12ptKDGycqnGzUZUz5Jy75"  # 大模型 API Key

    print("\n[System] 启动全市场因子挖掘环境...")
    
    # 1. 实例化 Tushare 数据加载器 (修改为 2025年6月1日 到 2025年9月30日)
    data_loader = TushareDataLoader(
        token=TUSHARE_TOKEN, 
        start_date="2020316", 
        end_date="20260316"
    )
    
    # 获取全市场 5000+ 股票的宽表矩阵
    unpost_stock_data,stock_data = data_loader.fetch_daily_data()
    benchmark_returns = data_loader.fetch_benchmark_returns(ts_code='000300.SH')
    # 收益率矩阵构建
    forward_returns = data_loader.build_forward_returns(stock_data['Close'], periods=1)

    print("\n[System] 初始化智能体依赖库...")
    
    # 2. 实例化记忆存储 (区分存储，避免与之前沪深300的混淆)
    info_memory = InformationBase("all_market_info_base.json")
    factor_assets = FactorBase("all_market_factor_base.json", "./all_market_factor_data/")

    # 3. 实例化评测引擎
    engine = EvaluationEngine(
        stock_data=stock_data, 
        forward_returns=forward_returns, 
        factor_base=factor_assets,
        benchmark_returns=benchmark_returns
    )

    # 4. 实例化因子挖掘 Agent 
    agent = FactorMiningAgent(
        info_base=info_memory, 
        factor_base=factor_assets, 
        eval_engine=engine, 
        api_key=OPENAI_API_KEY
    )

    print("\n[System] 数据加载完毕，开始在【全市场环境】执行 Ralph Loop 智能迭代因子挖掘...")
    
    # 5. 启动迭代
    try:
        agent.run_ralph_loop(max_iterations=2)
    except KeyboardInterrupt:
        print("\n[System] 收到中断信号，因子挖掘系统已安全停止。")


[System] 启动全市场因子挖掘环境...
命中本地缓存: ./data_cache/all_a_shares_20240601_20250930.parquet，正在使用 PyArrow 飞速加载全市场数据...
正在构建量化计算引擎数据矩阵 (长表转宽表)...

[System] 初始化智能体依赖库...

[System] 数据加载完毕，开始在【全市场环境】执行 Ralph Loop 智能迭代因子挖掘...

--- 启动第 1 轮迭代 ---
Agent 生成公式: Ts_Max(Rank(Delay(Delta(Close, 5), 2) / Ts_Mean(Volume, 20)), 10) - Abs(Correlation(Volume, Close, 10))
评测结果: 是否成功=True, 指标={'IC': 0.0307, 'Turnover': 0.1322, 'Max_Correlation': 0.0}
✅ 因子入库成功！ID 记录已更新。

--- 启动第 2 轮迭代 ---
Agent 生成公式: Rank(Ts_Max(Delta(Close, 3) * Volume / Ts_Mean(Abs(Delta(Close, 1)), 10), 5))
评测结果: 是否成功=True, 指标={'IC': -0.037, 'Turnover': 0.0872, 'Max_Correlation': np.float64(0.1362)}
✅ 因子入库成功！ID 记录已更新。

--- 启动第 3 轮迭代 ---
Agent 生成公式: Rank(Ts_Mean(Abs(Delta(Close, 3)) * Volume / Ts_Mean(Volume, 20), 10))
评测结果: 是否成功=True, 指标={'IC': -0.0412, 'Turnover': 0.0492, 'Max_Correlation': np.float64(0.1886)}
✅ 因子入库成功！ID 记录已更新。

--- 启动第 4 轮迭代 ---
Agent 生成公式: Rank(Correlation(Abs(Delta(Close, 2)), Ts_Mean(Volume, 10), 5) / Delta(Ts_Mean(Abs(De

RateLimitError: Error code: 429 - {'error': {'message': 'deepseek-chat模型免费API限制每日30次请求，请00:00后再试，如有更多需求，请访问 https://api.chatanywhere.tech/#/shop 购买付费API。The free account is limited to 30 requests per day. Please try again after 00:00 the next day. If you have additional requirements, please visit https://api.chatanywhere.tech/#/shop to purchase a premium key.(当前请求使用的ApiKey: sk-Cqq****Jy75)【如果您遇到问题，欢迎加入QQ群咨询：836739524】', 'type': 'chatanywhere_error', 'param': None, 'code': '429 TOO_MANY_REQUESTS'}}

In [ ]:
"""
可以优化的方向：
1.用更好的大语言模型
2.采用更多的大模型，分属不同职位
3.多模态输入
4.增添算子和机器学习算子
5.增添因子评价
6.优化prompt结构，采用cot
7.输入数据细粒度
8.算子gpu加速



"""

{'Open': ts_code     000001.SZ  000002.SZ  000004.SZ  000006.SZ  000007.SZ  000008.SZ  \
 trade_date                                                                     
 2025-09-29        NaN        NaN        NaN        NaN        NaN        NaN   
 2025-09-30      11.37        6.8      11.12       9.48        7.2       2.88   
 
 ts_code     000009.SZ  000010.SZ  000011.SZ  000012.SZ  ...  920964.BJ  \
 trade_date                                              ...              
 2025-09-29        NaN        NaN        NaN        NaN  ...       7.35   
 2025-09-30      12.62        4.3       9.01       4.68  ...       7.49   
 
 ts_code     920970.BJ  920971.BJ  920974.BJ  920976.BJ  920978.BJ  920981.BJ  \
 trade_date                                                                     
 2025-09-29       9.58      38.49       8.44      25.36      41.45      36.36   
 2025-09-30       9.60      39.95       8.65      26.31      40.30      37.36   
 
 ts_code     920982.BJ  920985.BJ  920